# 4.2 Mean-shift clustering

Mean-shift clustering turns unlabeled data into structure by choosing the right notion of similarity, compression, or surprise.

Probability normalization and distance-based kernels become a local movement rule. Points repeatedly shift toward weighted density modes, so bandwidth controls the scale of reality the algorithm can see.

Save a copy to Drive to edit. This notebook is deterministic, CPU-only, and uses only bundled scikit-learn data.

In [ ]:

import numpy as np
import matplotlib.pyplot as plt

from collections import deque
from sklearn.cluster import AgglomerativeClustering
from sklearn.cluster import AffinityPropagation
from sklearn.cluster import Birch
from sklearn.cluster import DBSCAN
from sklearn.cluster import KMeans
from sklearn.cluster import MeanShift
from sklearn.cluster import OPTICS
from sklearn.cluster import estimate_bandwidth
from sklearn.datasets import load_digits
from sklearn.datasets import load_iris
from sklearn.datasets import make_blobs
from sklearn.decomposition import PCA
from sklearn.metrics import adjusted_rand_score
from sklearn.metrics import pairwise_distances
from sklearn.metrics import silhouette_score
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler

SEED = 7
rng = np.random.default_rng(SEED)

def cluster_ladder():
    """D1..D5 clustering ladder of rising difficulty. Returns [(name, X, y_true, k), ...].

    y_true is the generating label (for ARI scoring only — clustering does not see it).
    Rungs: hand points -> clean blobs -> anisotropic/overlap -> real Iris -> real digits(4-class).
    """
    rungs = []

    x1 = np.array([[0.0, 0.0], [0.3, 0.2], [3.0, 3.0], [3.2, 2.8], [0.1, 3.1], [0.2, 2.9]])
    y1 = np.array([0, 0, 1, 1, 2, 2])
    rungs.append(("D1 hand 3 clusters", x1, y1, 3))

    x2, y2 = make_blobs(n_samples=200, centers=3, cluster_std=0.7, random_state=1)
    rungs.append(("D2 clean blobs", x2, y2, 3))

    x3, y3 = make_blobs(n_samples=240, centers=3, cluster_std=1.6, random_state=2)
    transform = np.array([[0.6, -0.6], [-0.4, 0.8]])
    x3 = x3 @ transform
    rungs.append(("D3 anisotropic + overlap", x3, y3, 3))

    iris = load_iris()
    rungs.append(("D4 Iris (real, 4-D)", iris.data, iris.target, 3))

    digits = load_digits()
    keep = np.isin(digits.target, [0, 1, 2, 3])
    rungs.append(("D5 digits 0-3 (real, 64-D)", digits.data[keep] / 16.0, digits.target[keep], 4))

    return rungs



def project_2d(X):
    X = np.asarray(X, dtype=float)
    if X.shape[1] == 2:
        return X
    return PCA(n_components=2, random_state=SEED).fit_transform(X)


def ari_score(y_true, labels):
    return float(adjusted_rand_score(y_true, labels))


def safe_silhouette(X, labels):
    labels = np.asarray(labels)
    keep = labels != -1
    unique = np.unique(labels[keep])
    if keep.sum() < 3:
        return np.nan
    if unique.size < 2:
        return np.nan
    if unique.size >= keep.sum():
        return np.nan
    return float(silhouette_score(X[keep], labels[keep]))


def preview_ladder(rungs):
    rows = []
    for idx, item in enumerate(rungs, start=1):
        name, X, y_true, k = item
        rows.append((idx, name, X.shape, int(np.unique(y_true).size), k, np.round(X[:3], 3).tolist()))
    return rows


def plot_cluster_panels(results, title, show_centers=False):
    fig, axes = plt.subplots(1, len(results), figsize=(18, 3.4))
    for ax, result in zip(axes, results):
        Z = project_2d(result["X"])
        ax.scatter(Z[:, 0], Z[:, 1], c=result["labels"], s=16, cmap="tab10", alpha=0.82)
        if show_centers and result.get("centers") is not None:
            centers = np.asarray(result["centers"])
            if len(centers) > 0:
                centers_2d = centers if centers.shape[1] == 2 else PCA(n_components=2, random_state=SEED).fit(result["X"]).transform(centers)
                ax.scatter(centers_2d[:, 0], centers_2d[:, 1], c="black", marker="x", s=70)
        ax.set_title(f"D{result['rung']} ARI={result['ari']:.2f}")
        ax.set_xticks([])
        ax.set_yticks([])
    fig.suptitle(title)
    plt.show()


def plot_ari_curve(results, title):
    xs = [result["rung"] for result in results]
    ys = [result["ari"] for result in results]
    plt.figure(figsize=(6, 3.5))
    plt.plot(xs, ys, marker="o")
    plt.ylim(-0.05, 1.05)
    plt.xlabel("D1 to D5 ladder rung")
    plt.ylabel("Adjusted Rand Index")
    plt.title(title)
    plt.grid(True, alpha=0.3)
    plt.show()


## 3. The concept, built once on D1

Mean-shift uses kernel weights $$p_i=rac{\exp(-d_i^2/2h^2)}{\sum_j\exp(-d_j^2/2h^2)}$$. With $h=1$ and distances $[1,0,1]$, the central weight is the lesson's $0.452$.

In [ ]:
# Formula audit: $p_i = \exp(-d_i^2 / 2h^2) / \sum_j \exp(-d_j^2 / 2h^2)$
h = 1.0
distances = np.array([1.0, 0.0, 1.0])
weights = np.exp(-(distances ** 2) / (2 * h ** 2))
rounded_weights = np.round(weights, 3).tolist()
weight_sum = float(np.round(weights.sum(), 3))
middle_weight = float(np.round(weights[1] / weights.sum(), 3))
assert h == 1.0
assert rounded_weights == [0.607, 1.0, 0.607]
assert weight_sum == 2.213
assert middle_weight == 0.452
print(np.round(weights, 3), weight_sum, middle_weight)

Now implement the real mode-seeking loop: every point repeatedly moves to the Gaussian-kernel weighted average of its neighbors, then nearby converged modes are merged.

In [ ]:
def merge_modes(shifted, bandwidth):
    centers = []
    labels = np.full(len(shifted), -1, dtype=int)
    for i, point in enumerate(shifted):
        assigned = False
        for j, center in enumerate(centers):
            if np.linalg.norm(point - center) <= bandwidth / 2:
                labels[i] = j
                centers[j] = 0.5 * center + 0.5 * point
                assigned = True
                break
        if not assigned:
            labels[i] = len(centers)
            centers.append(point.copy())
    return labels, np.vstack(centers)


def method(X, k=None, bandwidth=1.0, max_iter=40, tol=1e-3):
    X = np.asarray(X, dtype=float)
    shifted = X.copy()
    for _ in range(max_iter):
        previous = shifted.copy()
        for i, point in enumerate(previous):
            distances = np.linalg.norm(X - point, axis=1)
            weights = np.exp(-(distances ** 2) / (2 * bandwidth ** 2))
            shifted[i] = np.average(X, axis=0, weights=weights)
        movement = np.linalg.norm(shifted - previous, axis=1).max()
        if movement < tol:
            break
    labels, centers = merge_modes(shifted, bandwidth)
    return labels, centers

## 4. The dataset ladder

The same clustering function runs on D1 hand points, D2 clean blobs, D3 anisotropic overlap, D4 real Iris, and D5 real digits. Hidden labels are kept only for ARI scoring; the clustering method never receives them.

In [ ]:
rungs = cluster_ladder()
for row in preview_ladder(rungs):
    print(row)

## 5. Run the same method across D1-D5

The metric is Adjusted Rand Index against hidden generating labels. The labels are never passed into `method`; they are used only after clustering for evaluation.

In [ ]:
results = []
for rung, (name, X, y_true, k) in enumerate(rungs, start=1):
    X_use = StandardScaler().fit_transform(X)
    bandwidth = max(0.35, float(estimate_bandwidth(X_use, quantile=0.25, n_samples=min(300, len(X_use)))))
    labels, centers = method(X_use, bandwidth=bandwidth)
    score = ari_score(y_true, labels)
    sil = safe_silhouette(X_use, labels)
    results.append({"rung": rung, "name": name, "X": X_use, "labels": labels, "centers": centers, "ari": score, "silhouette": sil, "bandwidth": bandwidth})
    print(f"D{rung} {name}: ARI={score:.3f} silhouette={sil:.3f} modes={len(np.unique(labels))} bandwidth={bandwidth:.3f}")

## 6. Results visualization

Panels show mean-shift mode assignments. The curve tracks ARI across the ladder.

In [ ]:
plot_cluster_panels(results, "mean-shift cluster assignments", show_centers=True)
plot_ari_curve(results, "mean-shift ARI vs ladder rung")

## 7. Pitfall on D5: bandwidth instability

Mean-shift does not ask for $k$, but bandwidth can explode modes or collapse them. We reproduce both failures on digits, then sweep nearby bandwidths and prefer stable ARI and mode counts.

In [ ]:
name, X5, y5, k5 = rungs[-1]
X5s = StandardScaler().fit_transform(X5)
small_labels, small_centers = method(X5s, bandwidth=0.25)
large_labels, large_centers = method(X5s, bandwidth=4.0)
print("small bandwidth modes", len(np.unique(small_labels)), "ARI", round(ari_score(y5, small_labels), 3))
print("large bandwidth modes", len(np.unique(large_labels)), "ARI", round(ari_score(y5, large_labels), 3))
for bandwidth in [0.8, 1.2, 1.6, 2.0]:
    labels, centers = method(X5s, bandwidth=bandwidth)
    print("sweep", bandwidth, "modes", len(np.unique(labels)), "ARI", round(ari_score(y5, labels), 3))

## 8. Evaluate it + practice

- Metric: Adjusted Rand Index vs hidden labels, plus a no-skill sanity check from random labels.
- Sanity check: rerun with a nearby seed or hyperparameter and confirm the D5 story does not flip.
- Ablation: turn off the key idea in this lesson and watch ARI or stability drop.
- Failure signal: one cluster, all noise, or many singleton clusters usually means scale or hyperparameters dominate.
- D5 check: digits are real 64-D images, so visualization is a projection while clustering uses the full feature matrix.

Ablation: replace the Gaussian weights with equal weights inside the bandwidth and compare the D3 assignments.

Practice prompts:
1. Change one hyperparameter on D3 and explain whether ARI or the assignment panel changes first.

2. Add a stability rerun on D5 and compare the resulting ARI values.

3. Replace StandardScaler with no scaling on D4 or D5 and describe the failure mode.